#### LIBRARY IMPORTS

In [61]:
import pandas as pd
import numpy as np
import glob # For file pattern matching for loading files
import os # Certain debugging operations
import joblib # To save scaler and encoder

from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Neural Netowrk specifc imports
import copy # for deepcopy in save_results()

import torch
import torch.nn as nn # NN layers and loss functions
import torch.optim as optim # Optimization Algorithms
# Batching Data:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

#### CONFIGURATOINS

In [62]:
FEATURE_ROOT = "speaker_wise-eGeMAPs/functionals"
OUTPUT_ROOT = "model_outputs"
RANDOM_SEED = 37
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHILD_SPEAKER = {
     
    "speaker_functional_p5-s2.csv": "SPEAKER_00",
    "speaker_functional_p5-s7.csv": "SPEAKER_01",
    "speaker_functional_p5-s8.csv": "SPEAKER_05",
    "speaker_functional_p5-s10.csv": "SPEAKER_01",
    "speaker_functional_p5-s13.csv": "SPEAKER_01",

    "speaker_functional_p7-s5.csv": "SPEAKER_01",
    "speaker_functional_p7-s6.csv": "SPEAKER_08",
    "speaker_functional_p7-s7.csv": "SPEAKER_02",
    "speaker_functional_p7-s8.csv": "SPEAKER_00",
    "speaker_functional_p7-s16.csv": "SPEAKER_00",
    "speaker_functional_p7-s17.csv": "SPEAKER_00",
    "speaker_functional_p7-s18.csv": "SPEAKER_02",
    "speaker_functional_p7-s29.csv": "SPEAKER_00",

    "speaker_functional_p9-s3-1.csv": "SPEAKER_01",
    "speaker_functional_p9-s3-2.csv": "SPEAKER_04",
    "speaker_functional_p9-s4.csv": "SPEAKER_06",
    "speaker_functional_p9-s9.csv": "SPEAKER_03",
    "speaker_functional_p9-s15.csv": "SPEAKER_01",

    "speaker_functional_p11-s2.csv": "SPEAKER_02",
    "speaker_functional_p11-s4.csv": "SPEAKER_01",
    "speaker_functional_p11-s8.csv": "SPEAKER_03",
    "speaker_functional_p11-s9.csv": "SPEAKER_00",
    "speaker_functional_p11-s11.csv": "SPEAKER_05",
    "speaker_functional_p11-s15.csv": "SPEAKER_00",
    "speaker_functional_p11-s16-2.csv": "SPEAKER_01",
    "speaker_functional_p11-s19.csv": "SPEAKER_00",
    "speaker_functional_p11-s22-2.csv": "SPEAKER_02",

    "speaker_functional_p12-s2-2.csv": "SPEAKER_00",
    "speaker_functional_p12-s3.csv": "SPEAKER_01",
    "speaker_functional_p12-s6.csv": "SPEAKER_01",
    "speaker_functional_p12-s8.csv": "SPEAKER_00",
    "speaker_functional_p12-s10.csv": "SPEAKER_03",

    "speaker_functional_p17-s2.csv": "SPEAKER_01",
    "speaker_functional_p17-s3.csv": "SPEAKER_04",
    "speaker_functional_p17-s5.csv": "SPEAKER_04",
    "speaker_functional_p17-s6.csv": "SPEAKER_02",

    "speaker_functional_p18-s3.csv": "SPEAKER_00",
    "speaker_functional_p18-s4.csv": "SPEAKER_01",
    "speaker_functional_p18-s5.csv": "SPEAKER_00",
    "speaker_functional_p18-s7.csv": "SPEAKER_01",
    "speaker_functional_p18-s8.csv": "SPEAKER_00",
    "speaker_functional_p18-s9.csv": "SPEAKER_01",
    "speaker_functional_p18-s10.csv": "SPEAKER_06",
    "speaker_functional_p18-s11.csv": "SPEAKER_00",
    "speaker_functional_p18-s12.csv": "SPEAKER_00",
    "speaker_functional_p18-s13.csv": "SPEAKER_01",
    "speaker_functional_p18-s15.csv": "SPEAKER_02",
    "speaker_functional_p18-s17.csv": "SPEAKER_01",
    "speaker_functional_p18-s18.csv": "SPEAKER_00",
    "speaker_functional_p18-s19.csv": "SPEAKER_02",
    "speaker_functional_p18-s20.csv": "SPEAKER_01",

}

#### FUNCTIONS

In [63]:
# Early stopping
PATIENCE = 10 # Stop after 10 epochs of no improvement in validation loss

In [64]:
def load_participant_data(participant_folder):
    csv_files = sorted(glob.glob(f"{FEATURE_ROOT}/{participant_folder}/*.csv")) # For multiple files

    def load_single_speaker(csv_path):
        df = pd.read_csv(csv_path)    
        filename = os.path.basename(csv_path)

        if filename not in CHILD_SPEAKER:
            raise ValueError(
                f"No child speaker mapping for {filename}"
            )
        
        target_speaker = CHILD_SPEAKER[filename]

        df = df[df["speaker"] == target_speaker]

        return df
    
    # Train / Validation / Test Split
    
    print(f"{participant_folder}: {len(csv_files)} CSV files")
    
    
    # Load every session for this participant
    full_df = pd.concat(
        [
            load_single_speaker(f)
            for f in csv_files
        ],
        ignore_index=True
    )

    print(f"Total child utterances: {len(full_df)}")
    
    # Remove unnecessary columns
    drop_cols = [
        "participant",
        "session",
        "clip_id",
        "speaker",

        "engagement_start_time",
        "engagement_end_time",

        "speaker_start_time",
        "speaker_end_time",

        "num_segments",
        "speech_duration",
    ]

    full_df = full_df.drop(
        columns=[c for c in drop_cols if c in full_df.columns]
    )

    print(full_df.columns)

    # Separate Features and Labels
    X = full_df.drop(columns=["label"])
    y = full_df["label"]

    return (
        X.reset_index(drop=True),
        y.reset_index(drop=True)
    )

In [65]:
def preprocess_data(X_train, X_val, X_test, y_train, y_val, y_test):
    
    # Label Encoding
    encoder = LabelEncoder()
    y_train = encoder.fit_transform(y_train)
    y_val = encoder.transform(y_val)
    y_test = encoder.transform(y_test)

    # Print for debugging purpose
    print(encoder.classes_)
    print(np.unique(y_train))
    print(np.unique(y_val))
    print(np.unique(y_test))

    # Feature Scaling
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train) # Learn the Meand and std and transform the data
    X_val = scaler.transform(X_val) # Transform the data using the learned mean and std
    X_test = scaler.transform(X_test) # Transform the data using the learned mean and std

    return (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
        scaler,
        encoder
    )  

In [66]:
class NeuralNetwork(nn.Module):

    # Define the architecture of the neural network ; Constructor
    def __init__(self, input_dim):

        super().__init__() # Initialize the parent class (nn.Module) first, then inherit functionalities

        self.network = nn.Sequential(
            # First Hidden Layer 88 -> 128 nueruons
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # Second Hidden Layer 128 -> 64 nueruons
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Output Layer 64 -> 1 nuerons
            nn.Linear(64, 1)

        )
    
    # Forward pass through the network; Automatically called  when calling model; Then return the model predictions
    def forward(self, x):
        return self.network(x)


# Build the model, then move it to GPU/ CPU and print the model architecture
def build_model(input_dim):
    # Build model
    model = NeuralNetwork(input_dim)
    
    # Move model to GPU/ CPU
    model.to(DEVICE)
    
    print(model)

    return model

In [67]:
def train_model(
        model,
        train_loader,
        val_loader
):
    # BCEWithLogitsLoss is good for our case of binary classification
    # It performs sigmoid + Binary Cross Entropy Loss 
    criterion = nn.BCEWithLogitsLoss()

    # Adam optimizer
    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    # Initialize best loss to be infinity and patience counter to 0
    best_loss = float("inf")
    patience_counter = 0

    # Store training and validation loss for each epoch (perhaps for plotting later)
    history = {
        "train_loss": [],
        "val_loss": []
    }

    # Epoch Loop; Max is 100, but may stop earlier due to early stopping
    for epoch in range(100):

        #TRAINING

        # Activate training (Dropout layers and gradient computation)
        model.train()

        train_loss = 0

        # Train Batch-wise (32)
        for X_batch, y_batch in train_loader:
            
            # Move batch to GPU/ CPU
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            # Reset the gradients before backpropagation
            optimizer.zero_grad()

            # Foeward pass prediction
            outputs = model(X_batch)

            # Compute loss
            loss = criterion(outputs, y_batch)

            # Backpropagation 
            # Computes gradients
            loss.backward()
            # Update weights using optimizer
            optimizer.step()

            train_loss += loss.item() # loss is a tensor

        # Average traingin loss per batch
        train_loss /= len(train_loader)

        # VALIDATION
        
        # Turn off training (Dropout layers and gradient computation)
        model.eval()

        val_loss = 0

        # Disable gradient computation
        with torch.no_grad():
            # Validate Batch-wise (32)
            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)

                outputs = model(X_batch)

                loss = criterion(outputs, y_batch)

                val_loss += loss.item()

        val_loss /= len(val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch+1} "
            f"Train={train_loss:.4f} "
            f"Val={val_loss:.4f}"
        )

        # If validation loss improves, save the model weights and reset patience counter
        if val_loss < best_loss:
            best_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0

        # If validation loss does not improve, increment patience counter and check for early stopping
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping")
                break

    # Restore the best model weights after training is complete
    model.load_state_dict(best_weights)

    return history

In [68]:
def evaluate_model(
        model,
        test_loader,
        encoder
):
    # No dropouts
    model.eval()

    probabilities = []
    predictions = []
    actual = []

    # Disable gradient computation
    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(DEVICE)

            # Forward pass prediction
            outputs = model(X_batch)
            
            # Apply sigmoid to get probabilities
            probs = torch.sigmoid(outputs)

            # Convert probabilities to binary predictions (0 or 1) using a threshold of 0.5
            preds = (probs >= 0.5).float()

            # NumPy cannotread GPU tensors, so we need to move them to CPU and convert to NumPy arrays before storing them in lists
            probabilities.extend(probs.cpu().numpy().flatten())
            predictions.extend(preds.cpu().numpy().flatten())
            
            actual.extend(y_batch.numpy().flatten())

    # Convert lists to NumPy arrays and ensure they are of integer type
    predictions = np.array(predictions).astype(int)
    actual = np.array(actual).astype(int)

    # Confusion Matrix
    cm = confusion_matrix(actual, predictions)

    # Compute metrics
    metrics_dictionary = {
        "Accuracy": accuracy_score(actual, predictions),
        "Precision": precision_score(actual, predictions),
        "Recall": recall_score(actual, predictions),
        "F1 Score": f1_score(actual, predictions)
    }

    report = classification_report(
        actual,
        predictions,
        target_names=encoder.classes_,
        output_dict=True
    )

    print(classification_report(
        actual,
        predictions,
        target_names=encoder.classes_
    ))

    return {
        "metrics": metrics_dictionary,
        "predictions": predictions,
        "probabilities": probabilities,
        "actual": actual,
        "confusion_matrix": cm,
        "classification_report": report
    }

In [69]:
''' The following are saved:
    Model weights (model.pt)
    Scaler (scaler.pkl)
    Encoder (encoder.pkl)
    Training History (history.csv)
    Metrics (metrics.csv)
    Confusion Matrix (confusion_matrix.csv)
    Classification Report (classification_report.csv) '''
    
def save_results(
        participant,
        model,
        scaler,
        encoder,
        history,
        evaluation
    ):
    
    metrics = evaluation["metrics"]
    predictions = evaluation["predictions"]
    probabilities = evaluation["probabilities"]
    actual = evaluation["actual"]
    cm = evaluation["confusion_matrix"]
    report = evaluation["classification_report"]
    
    # Save Model
    participant_output = Path(OUTPUT_ROOT, participant)
    os.makedirs(participant_output, exist_ok=True)
    torch.save(model.state_dict(),participant_output / "model.pt")
    
    # Save Scaler
    joblib.dump(scaler,Path(participant_output, "scaler.pkl"))
    
    # Save Encoder
    joblib.dump(encoder,Path(participant_output, "encoder.pkl"))
    
    # Save History
    history_df = pd.DataFrame(history)
    history_df.to_csv(Path(participant_output, "history.csv"),index=False)
    
    # Save Metrics
    metrics_df = pd.DataFrame([metrics])

    metrics_df.to_csv(Path(participant_output, "metrics.csv"),index=False)
        
    # Save Confusion Matrix
    cm_df = pd.DataFrame(cm,index=encoder.classes_,columns=encoder.classes_)
    cm_df.to_csv(Path(participant_output, "confusion_matrix.csv"))
    
    # Save classification report
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(Path(participant_output, "classification_report.csv"))
    
    # Save predictions
    prediction_df = pd.DataFrame({
        "Actual": actual,
        "Prediction": predictions,
        "Probability": probabilities
    })

    prediction_df.to_csv(Path(participant_output, "predictions.csv"),index=False)

#### RUN MODEL

In [70]:
summary_results = []

participants = sorted(os.listdir(FEATURE_ROOT))

all_fold_results = []

for participant in participants:

    print(f"Training {participant}")

    # Load ALL data for this participant
    X, y = load_participant_data(participant)

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_SEED
    )

    participant_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

        print(f"\nFold {fold}/5")

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        X_train, X_val, y_train, y_val = train_test_split(
            X_train,
            y_train,
            test_size=0.20,
            stratify=y_train,
            random_state=RANDOM_SEED
        )

        # Preprocess data
        X_train, X_val, X_test, y_train, y_val, y_test, scaler, encoder = preprocess_data(X_train, X_val, X_test, y_train, y_val, y_test)

        # Convert NumPy arrays into PyTorch tensors.
        X_train_tensor = torch.FloatTensor(X_train)
        X_val_tensor = torch.FloatTensor(X_val)
        X_test_tensor = torch.FloatTensor(X_test)

        # Shape of y_train, y_val, y_test is (N,), but we need (N,1) for BCEWithLogitsLoss
        y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
        y_val_tensor = torch.FloatTensor(y_val).unsqueeze(1)
        y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

        # Create TensorDatasets; Join features + labels
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
        test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

        # Create DataLoaders; Batch the data
        train_loader = DataLoader(
            train_dataset,
            batch_size=32,
            shuffle=True # Shuffle training data for better generalization
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=32,
            shuffle=False
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=32,
            shuffle=False
        )

        # Build model
        model = build_model(X_train.shape[1])

        # Train model and return training history
        history = train_model(model, train_loader, val_loader)

        # Evaluate model and return evaluation metrics
        evaluation = evaluate_model(model, test_loader, encoder)

        # Append metrics for this participant to the summary results
        participant_metrics.append(evaluation["metrics"])

        # Append fold results to all_fold_results
        all_fold_results.append({
            "Participant": participant,
            "Fold": fold,
            **evaluation["metrics"]
        })
        
        # Save results
        save_results(
            f"{participant}/fold_{fold}",
            model,
            scaler,
            encoder,
            history,
            evaluation
        )
    
    metrics_df = pd.DataFrame(participant_metrics)
    
    summary_results.append({
        "Participant": participant,

        "Accuracy Mean": metrics_df["Accuracy"].mean(),
        "Accuracy Std": metrics_df["Accuracy"].std(),

        "Precision Mean": metrics_df["Precision"].mean(),
        "Precision Std": metrics_df["Precision"].std(),

        "Recall Mean": metrics_df["Recall"].mean(),
        "Recall Std": metrics_df["Recall"].std(),

        "F1 Mean": metrics_df["F1 Score"].mean(),
        "F1 Std": metrics_df["F1 Score"].std()
    })
      
pd.DataFrame(all_fold_results).to_csv(
    "fold_results.csv",
    index=False
)

# Save summary results as CSV
summary_df = pd.DataFrame(summary_results)
summary_df.to_csv(
    Path(OUTPUT_ROOT,"summary_results.csv"),
    index=False
)
    

Training p11
p11: 9 CSV files
Total child utterances: 272
Index(['F0semitoneFrom27.5Hz_sma3nz_amean',
       'F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
       'F0semitoneFrom27.5Hz_sma3nz_percentile20.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile50.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile80.0',
       'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2',
       'F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_meanFallingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevFallingSlope', 'loudness_sma3_amean',
       'loudness_sma3_stddevNorm', 'loudness_sma3_percentile20.0',
       'loudness_sma3_percentile50.0', 'loudness_sma3_percentile80.0',
       'loudness_sma3_pctlrange0-2', 'loudness_sma3_meanRisingSlope',
       'loudness_sma3_stddevRisingSlope', 'loudness_sma3_meanFallingSlope',
       'loudness_sma3_stddevFallingSlope', 'spectralFlux_sma3_amean',
       'spectralFlux_sma3_stddevNorm', '

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

Epoch 20 Train=0.3042 Val=0.7390
Early stopping
              precision    recall  f1-score   support

  disengaged       0.50      0.37      0.42        19
     engaged       0.70      0.80      0.75        35

    accuracy                           0.65        54
   macro avg       0.60      0.58      0.59        54
weighted avg       0.63      0.65      0.63        54


Fold 4/5
['disengaged' 'engaged']
[0 1]
[0 1]
[0 1]
NeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=88, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)
Epoch 1 Train=0.6907 Val=0.6598
Epoch 2 Train=0.6659 Val=0.6466
Epoch 3 Train=0.6521 Val=0.6346
Epoch 4 Train=0.6259 Val=0.6233
Epoch 5 Train=0.6131 Val=0.6103
Epoch 6 Train=0.5781 Val=0.5977
Epoch 7 Train=0.5965 Val=0.5859
Epoch 8 Tra

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

Epoch 19 Train=0.3755 Val=0.6260
Epoch 20 Train=0.3635 Val=0.6464
Epoch 21 Train=0.3455 Val=0.6606
Epoch 22 Train=0.3253 Val=0.6705
Epoch 23 Train=0.3865 Val=0.7025
Early stopping
              precision    recall  f1-score   support

  disengaged       0.33      0.18      0.23        17
     engaged       0.67      0.83      0.74        35

    accuracy                           0.62        52
   macro avg       0.50      0.50      0.49        52
weighted avg       0.56      0.62      0.58        52

Training p17
p17: 4 CSV files
Total child utterances: 81
Index(['F0semitoneFrom27.5Hz_sma3nz_amean',
       'F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
       'F0semitoneFrom27.5Hz_sma3nz_percentile20.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile50.0',
       'F0semitoneFrom27.5Hz_sma3nz_percentile80.0',
       'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2',
       'F0semitoneFrom27.5Hz_sma3nz_meanRisingSlope',
       'F0semitoneFrom27.5Hz_sma3nz_stddevRisingSlope',
       'F0semitoneFrom2

...

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

Epoch 22 Train=0.3848 Val=0.7086
Epoch 23 Train=0.4006 Val=0.7296
Early stopping
              precision    recall  f1-score   support

  disengaged       0.00      0.00      0.00         5
     engaged       0.69      1.00      0.81        11

    accuracy                           0.69        16
   macro avg       0.34      0.50      0.41        16
weighted avg       0.47      0.69      0.56        16


Fold 4/5
['disengaged' 'engaged']
[0 1]
[0 1]
[0 1]
NeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=88, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)
Epoch 1 Train=0.7220 Val=0.7173
Epoch 2 Train=0.7072 Val=0.6995
Epoch 3 Train=0.6799 Val=0.6835
Epoch 4 Train=0.6772 Val=0.6697
Epoch 5 Train=0.6685 Val=0.6562
Epoch 6 Train=0.6466 Val=0.6430
Epoch 7 Tr

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

Epoch 1 Train=0.6885 Val=0.6939
Epoch 2 Train=0.6778 Val=0.6931
Epoch 3 Train=0.6754 Val=0.6994
Epoch 4 Train=0.6502 Val=0.7055
Epoch 5 Train=0.6338 Val=0.7117
Epoch 6 Train=0.6043 Val=0.7206
Epoch 7 Train=0.6031 Val=0.7292
Epoch 8 Train=0.5820 Val=0.7394
Epoch 9 Train=0.5670 Val=0.7514
Epoch 10 Train=0.5623 Val=0.7644
Epoch 11 Train=0.5549 Val=0.7757
Epoch 12 Train=0.5198 Val=0.7872
Early stopping
              precision    recall  f1-score   support

  disengaged       0.00      0.00      0.00         7
     engaged       0.53      0.89      0.67         9

    accuracy                           0.50        16
   macro avg       0.27      0.44      0.33        16
weighted avg       0.30      0.50      0.38        16


Fold 3/5
['disengaged' 'engaged']
[0 1]
[0 1]
[0 1]
NeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=88, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
 

/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/interactionlab/anaconda3/envs/model_env/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p